In [ ]:
import json
import sys

from pathlib import Path
from datetime import datetime, timezone

import websocket

# ==============================================================================
# Import Protobuf
# ==============================================================================

PROTO_PATH = Path(
    r"D:\Data Projects\MEXC API Architecture\websocket-proto"
)

sys.path.append(str(PROTO_PATH))

from PushDataV3ApiWrapper_pb2 import PushDataV3ApiWrapper

# ==============================================================================
# WebSocket Configuration
# ==============================================================================

WS_URL = "wss://wbs-api.mexc.com/ws"

SYMBOL = "ETHUSDT"

ENDPOINT = f"spot@public.aggre.depth.v3.api.pb@10ms@{SYMBOL}"

subscription = {
    "method": "SUBSCRIPTION",
    "params": [
        ENDPOINT
    ],
    "id": 1
}

received_time = lambda: datetime.now(
    timezone.utc
).strftime(
    "%Y-%m-%d %H:%M:%S.%f UTC"
)

# ==============================================================================
# Connect
# ==============================================================================

ws = websocket.create_connection(WS_URL)

ws.send(
    json.dumps(subscription)
)

# ==============================================================================
# Local Order Book
# ==============================================================================

bid_book = {}
ask_book = {}

# ==============================================================================
# Receive Loop
# ==============================================================================

while True:

    message = ws.recv()

    if isinstance(message, str):
        continue

    wrapper = PushDataV3ApiWrapper()

    wrapper.ParseFromString(message)

    orderbook = wrapper.publicAggreDepths

    receive_timestamp = received_time()

    exchange_timestamp = datetime.fromtimestamp(
        wrapper.sendTime / 1000,
        tz=timezone.utc
    )

    # ==========================================================================
    # Update Local Bid Book
    # ==========================================================================

    for bid in orderbook.bids:

        price = float(bid.price)
        quantity = float(bid.quantity)

        if quantity == 0:
            bid_book.pop(price, None)
        else:
            bid_book[price] = quantity

    # ==========================================================================
    # Update Local Ask Book
    # ==========================================================================

    for ask in orderbook.asks:

        price = float(ask.price)
        quantity = float(ask.quantity)

        if quantity == 0:
            ask_book.pop(price, None)
        else:
            ask_book[price] = quantity

    # ==========================================================================
    # Liquidity Statistics
    # ==========================================================================

    total_bid_quantity = sum(
        bid_book.values()
    )

    total_bid_value = sum(
        price * quantity
        for price, quantity in bid_book.items()
    )

    total_ask_quantity = sum(
        ask_book.values()
    )

    total_ask_value = sum(
        price * quantity
        for price, quantity in ask_book.items()
    )

    # ==========================================================================
    # Best Bid / Best Ask
    # ==========================================================================

    best_bid = max(bid_book.keys()) if bid_book else None
    best_ask = min(ask_book.keys()) if ask_book else None

    spread = (
        best_ask - best_bid
        if best_bid is not None and best_ask is not None
        else None
    )

    imbalance = (
        total_bid_value /
        (total_bid_value + total_ask_value)
        if (total_bid_value + total_ask_value) > 0
        else 0
    )

    # ==========================================================================
    # Largest Bid Wall
    # ==========================================================================

    if bid_book:

        bid_wall_price, bid_wall_qty = max(
            bid_book.items(),
            key=lambda x: x[1]
        )

        bid_wall_value = bid_wall_price * bid_wall_qty

    else:

        bid_wall_price = None
        bid_wall_qty = 0
        bid_wall_value = 0

    # ==========================================================================
    # Largest Ask Wall
    # ==========================================================================

    if ask_book:

        ask_wall_price, ask_wall_qty = max(
            ask_book.items(),
            key=lambda x: x[1]
        )

        ask_wall_value = ask_wall_price * ask_wall_qty

    else:

        ask_wall_price = None
        ask_wall_qty = 0
        ask_wall_value = 0

    # ==========================================================================
    # Display
    # ==========================================================================

    print("\n" + "=" * 90)
    print("LIVE ORDER BOOK")
    print("=" * 90)

    print(f"Receive Time  : {receive_timestamp}")
    print(f"Exchange Time : {exchange_timestamp}")
    print(f"Version       : {orderbook.fromVersion} -> {orderbook.toVersion}")

    # ==========================================================================
    # Bids
    # ==========================================================================

    print("\n" + "=" * 90)
    print("CURRENT BIDS")
    print("=" * 90)

    print(
        f"{'Price':>15}"
        f"{'Quantity':>18}"
        f"{'Value (USDT)':>20}"
    )

    for price in sorted(
        bid_book.keys(),
        reverse=True
    ):

        quantity = bid_book[price]

        value = price * quantity

        print(
            f"{price:>15.2f}"
            f"{quantity:>18.5f}"
            f"{value:>20,.2f}"
        )

    # ==========================================================================
    # Asks
    # ==========================================================================

    print("\n" + "=" * 90)
    print("CURRENT ASKS")
    print("=" * 90)

    print(
        f"{'Price':>15}"
        f"{'Quantity':>18}"
        f"{'Value (USDT)':>20}"
    )

    for price in sorted(
        ask_book.keys()
    ):

        quantity = ask_book[price]

        value = price * quantity

        print(
            f"{price:>15.2f}"
            f"{quantity:>18.5f}"
            f"{value:>20,.2f}"
        )

    # ==========================================================================
    # Liquidity Summary
    # ==========================================================================

    print("\n" + "=" * 90)
    print("LIQUIDITY SUMMARY")
    print("=" * 90)

    print(f"Current Bid Liquidity : {total_bid_value:,.2f} USDT")
    print(f"Current Ask Liquidity : {total_ask_value:,.2f} USDT")
    print(f"Current Imbalance     : {imbalance:.2%}")

    if spread is not None:
        print(f"Current Spread        : {spread:.2f}")

    print()

    if bid_wall_price is not None:

        print(
            f"Largest Bid Wall      : "
            f"{bid_wall_price:.2f} | "
            f"{bid_wall_qty:,.5f} ETH | "
            f"{bid_wall_value:,.2f} USDT"
        )

    if ask_wall_price is not None:

        print(
            f"Largest Ask Wall      : "
            f"{ask_wall_price:.2f} | "
            f"{ask_wall_qty:,.5f} ETH | "
            f"{ask_wall_value:,.2f} USDT"
        )

    print("=" * 90)